<a href="https://colab.research.google.com/github/Ettalibi11/Spark_labs/blob/lab00/spark_tp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step 1: Install PySpark & Start **Session**

# 1. Install PySpark

In [2]:
!pip install -q pyspark

# 2. Import SparkSession

In [3]:
from pyspark.sql import SparkSession

# 3. Create a Spark Session

In [5]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab0-OnlineRetail-Warmup")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

In [6]:
print("Spark version:", spark.version)

Spark version: 3.5.1


#Step 2: Get and Prepare the Data

In [7]:
!unzip -o OnlineRetail.csv.zip

Archive:  OnlineRetail.csv.zip
  inflating: OnlineRetail.csv        


#Step 3: Define Schema and Load Data

In [8]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, TimestampType

# 1. Define the specific types for every column

In [9]:
online_retail_schema = StructType([
    StructField("InvoiceNo", IntegerType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice", FloatType(), True),
    StructField("CustomerId", IntegerType(), True),
    StructField("Country", StringType(), True),
])

## 2. Load the CSV using that schema

In [10]:
df = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "M/d/yyyy H:m")
    .schema(online_retail_schema)
    .csv("OnlineRetail.csv")
)

In [11]:
print("Rows:", df.count())
df.printSchema()

Rows: 541909
root
 |-- InvoiceNo: integer (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: float (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Country: string (nullable = true)



In [12]:
# Show a few rows
df.show(5, truncate=False)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerId|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850     |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
+---------+-----

In [13]:
# Display columns
print("Columns:", df.columns)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerId', 'Country']


In [14]:
# Quick statistical summary on numeric fields
df.describe(["Quantity", "UnitPrice"]).show()

+-------+------------------+-----------------+
|summary|          Quantity|        UnitPrice|
+-------+------------------+-----------------+
|  count|            541909|           541909|
|   mean|  9.55224954743324|4.611113614622466|
| stddev|218.08115785023486|96.75985330031472|
|    min|            -80995|        -11062.06|
|    max|             80995|          38970.0|
+-------+------------------+-----------------+



#Step 4: Select and Transform Columns

In [15]:
from pyspark.sql.functions import col, expr

In [16]:
# Select specific columns
df.select("Country").show(5)
df.select("StockCode", "Description", "UnitPrice").show(5, truncate=False)

+--------------+
|       Country|
+--------------+
|United Kingdom|
|United Kingdom|
|United Kingdom|
|United Kingdom|
|United Kingdom|
+--------------+
only showing top 5 rows

+---------+-----------------------------------+---------+
|StockCode|Description                        |UnitPrice|
+---------+-----------------------------------+---------+
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER |2.55     |
|71053    |WHITE METAL LANTERN                |3.39     |
|84406B   |CREAM CUPID HEARTS COAT HANGER     |2.75     |
|84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|3.39     |
|84029E   |RED WOOLLY HOTTIE WHITE HEART.     |3.39     |
+---------+-----------------------------------+---------+
only showing top 5 rows



In [17]:
# Add a 'Flag' column (High Value Items)
df_flagged = df.selectExpr(
    "*",
    "UnitPrice > 100 as HighValueItem"
)
df_flagged.select("Description", "UnitPrice", "HighValueItem").show(5)

# Calculate Revenue (Quantity * Price) and Rename
df_with_value = df.withColumn("InvoiceValue", col("UnitPrice") * col("Quantity"))

+--------------------+---------+-------------+
|         Description|UnitPrice|HighValueItem|
+--------------------+---------+-------------+
|WHITE HANGING HEA...|     2.55|        false|
| WHITE METAL LANTERN|     3.39|        false|
|CREAM CUPID HEART...|     2.75|        false|
|KNITTED UNION FLA...|     3.39|        false|
|RED WOOLLY HOTTIE...|     3.39|        false|
+--------------------+---------+-------------+
only showing top 5 rows



In [18]:
# Rename the new column to 'LineTotal'
df_line_total = df_with_value.withColumnRenamed("InvoiceValue", "LineTotal")

df_line_total.select("Description", "LineTotal").show(5)

+--------------------+---------+
|         Description|LineTotal|
+--------------------+---------+
|WHITE HANGING HEA...|15.299999|
| WHITE METAL LANTERN|    20.34|
|CREAM CUPID HEART...|     22.0|
|KNITTED UNION FLA...|    20.34|
|RED WOOLLY HOTTIE...|    20.34|
+--------------------+---------+
only showing top 5 rows



#Step 5: Clean Up (Drop Columns)

In [19]:
# Drop CustomerId and StockCode
df_reduced = df.drop("CustomerId", "StockCode")

print("Original column count:", len(df.columns))
print("New column count:", len(df_reduced.columns))

Original column count: 8
New column count: 6


#Step 6: Aggregations (GroupBy)

In [20]:
from pyspark.sql.functions import avg, stddev_pop

In [21]:
# Average quantity per Country
avg_qty_per_country = (
    df.groupBy("Country")
    .agg(avg("Quantity").alias("avg_quantity"))
)
avg_qty_per_country.show(5)

+---------+------------------+
|  Country|      avg_quantity|
+---------+------------------+
|   Sweden| 77.13636363636364|
|Singapore| 22.85589519650655|
|  Germany|12.369457609268036|
|   France| 12.91106696272058|
|   Greece|10.657534246575343|
+---------+------------------+
only showing top 5 rows



In [22]:
# Statistics per Invoice
invoice_stats = (
    df.groupBy("InvoiceNo")
    .agg(
        avg("Quantity").alias("avg_quantity"),
        stddev_pop("Quantity").alias("std_quantity")
    )
)

In [23]:
invoice_stats.show(5)


+---------+------------------+------------------+
|InvoiceNo|      avg_quantity|      std_quantity|
+---------+------------------+------------------+
|   536532| 25.36986301369863|16.850272831671976|
|   537632|               1.0|               0.0|
|   538708| 10.61111111111111| 7.150282736359209|
|   538877|14.258278145695364| 27.56989037543246|
|   538993| 9.333333333333334| 2.748737083745107|
+---------+------------------+------------------+
only showing top 5 rows



#Step 7: Performance Comparison (The "Reflection" Part)

In [25]:
# Test 1: Explicit Schema (Fast)
print("--- Explicit Schema ---")


--- Explicit Schema ---


In [26]:
%%time
df_schema = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "M/d/yyyy H:m")
    .schema(online_retail_schema) # We tell Spark exactly what the types are
    .csv("OnlineRetail.csv")
)

CPU times: user 3.86 ms, sys: 992 µs, total: 4.85 ms
Wall time: 51.7 ms


In [27]:
# Test 2: Inferred Schema (Slow)
print("\n--- Inferred Schema ---")


--- Inferred Schema ---


In [28]:
%%time
df_infer = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "M/d/yyyy H:m")
    .option("inferSchema", "true") # Spark has to scan the whole file to guess types
    .csv("OnlineRetail.csv")
)

CPU times: user 2.96 ms, sys: 1.14 ms, total: 4.1 ms
Wall time: 5.82 s


In [29]:
spark.stop()